In [1]:
# Required Python packages
from datetime import date, datetime, timedelta
import calendar
import pandas as pd
import numpy as np
import itertools
from scipy.stats import norm
from scipy.optimize import curve_fit
import copy
import os

#Import Date opertative user defined functions
from ipynb.fs.full.user_defined_vik_functions import find_last_thurs_date_of_month, date_of_prev_thurs, \
                                                     prev_weekday, prev_workday_if_holiday, \
                                                     next_working_day_incl_input_date, next_weekday, \
                                                     next_weekday_incl_input_date, \
                                                     get_all_monthly_option_expiries, find_wkly_expries

#Import pricing functions
from ipynb.fs.full.user_defined_vik_functions import BSM_call, BSM_put

#Import data loading functions
from ipynb.fs.full.user_defined_vik_functions import load_all_mthly_data, find_latest_available_mibor

# Import dataframe naming functions 
from ipynb.fs.full.user_defined_vik_functions import get_mthly_df_name_from_expiry

#Path Settings
source_path = "/home/jupyter-partha/Vikranth - Chapter 2/"
input_sub_path = "Input Data/mkt_data_covid_region/"
output_sub_path = "Output Data/"
input_data_path = source_path + input_sub_path
output_data_path = source_path + output_sub_path


In [2]:
#Index of interest
stock_ident = "BANKNIFTY"

# Periods of interest will be a dictionary
#Key is the year, value is a list of months 1-12, 1- Jan, 2 - Feb,...12 - Dec
# For E.g., periods_of_interest = {2020: [3], 2019: [11, 12]}
periods_of_interest = {2020: [3]}

#List oh holidays
holidays_list = [date(2020, 3, 10)]

#Product to hedge
prod_type = "CE"


In [3]:
def fitting_func(X, deta_0, deta_1, deta_2, lambda_0, lambda_1, lambda_2):
    M,T = X
    T_r = np.reciprocal(T)
    
    return deta_0 * np.power(T_r, lambda_0) + \
           np.multiply(deta_1 * np.power(T_r, lambda_1), M) + \
           np.multiply(deta_2 * np.power(T_r, lambda_2),  (M**2))

def fit_quadr_function_to_impl_vol_surf(dict_wkly_expiries_each_month, output_path, stock_ident, prod_type="CE"):
    params_dict = {}
    for each_mnth in dict_wkly_expiries_each_month.keys():
        df_impl_vol_surf = pd.read_csv(output_path + "A1_" + stock_ident + "_" "Daily_vol_smile_" + prod_type + "_" + each_mnth + ".csv")
        wk_params = []
        for each_week in dict_wkly_expiries_each_month[each_mnth]:
            df_vol_surf = df_impl_vol_surf[df_impl_vol_surf["Date"] == each_week.strftime("%d-%b-%Y")]
            x1 = df_vol_surf[["Moneyness"]].to_numpy().reshape(-1)
            x2 = df_vol_surf[["T"]].to_numpy().reshape(-1)
            y = df_vol_surf[["Impl_Vol"]].to_numpy().reshape(-1)
            params, param_var = curve_fit(fitting_func, (x1,x2), y, p0=[1, -1, 0.25, 0.25, 0.25, 0.25], \
                                          maxfev=150000)
            wk_params.append(params)
        params_dict[each_mnth] = wk_params
    return params_dict

#list of params is a python list = [deta_0, deta_1, deta_2, lambda_0, lambda_1, lambda_2]
# s - spot
def generate_greeks_for_local_vol(s, strikes_list, T_array, list_of_params, prod_type ="CE"):
    deta_0, deta_1, deta_2, lambda_0, lambda_1, lambda_2 = list_of_params
    greeks_list = []
    i=0
    for T in T_array:
        strikes_array = strikes_list[i]
        i = i+1
        K_r = np.reciprocal(strikes_array.astype(float))
        dvol_by_dk = -1.0 * ( deta_1 * s * (T ** lambda_1) * np.power(K_r, 2) + \
                           2.0 * deta_2 * (s**2) * (T ** lambda_2) * np.power(K_r,3) )
    
        
        dvol_by_dksq = 2.0 * deta_1 * s * T ** lambda_1 * np.power(K_r, 3) + \
                       6.0 * deta_2 * (s**2) * (T ** lambda_2)* np.power(K_r,4)

        dvol_by_dT = - 1.0 * ( lambda_0 * deta_0 * (T ** (lambda_0+1)) + \
                            lambda_1 * deta_1 * (T ** (lambda_1+1)) * s * K_r + \
                            lambda_2 * deta_2 * (T ** (lambda_2+1)) * (s**2) * np.power(K_r,2))
        
        greek_T = {"dvol_by_dk": dvol_by_dk, "dvol_by_dksq":dvol_by_dksq, "dvol_by_dT":dvol_by_dT}

        greeks_list.append(greek_T)
        
    return greeks_list

    
def generate_local_vol_from_impl_vol(imp_vol, dvol_by_dk, dvol_by_dksq, dvol_by_dT, K, s, T, r):
    d_1_c = (1 / (imp_vol * (T ** 0.5))) 
    d1_log =  (np.log(s/K) + ((r + (imp_vol ** 2) / 2) * T))
    d1 = d_1_c * d1_log
    
    loc_var = ((imp_vol**2) + (2*T*imp_vol*dvol_by_dT) + (2*r*K*T*imp_vol*dvol_by_dk)) / \
              ( np.square(1 + K*d1*np.sqrt(T)*dvol_by_dk) + \
               np.square(K)*T*imp_vol* (dvol_by_dksq - d1*np.sqrt(T)* np.square(dvol_by_dk)) )
    local_vol = np.sqrt(loc_var)
    return local_vol
 


In [4]:
# Generate Local Volatility Surface

mthly_expiries_list = get_all_monthly_option_expiries(periods_of_interest, holidays_list)
dict_wkly_expiries_each_month = find_wkly_expries(mthly_expiries_list, holidays_list)
params_dict = fit_quadr_function_to_impl_vol_surf(dict_wkly_expiries_each_month, output_data_path, stock_ident, prod_type="CE")
df_futures_mkt = load_all_mthly_data(mthly_expiries_list, input_data_path, prod_type_lists=["FUT"], stock_ident=stock_ident)


for each_month in dict_wkly_expiries_each_month.keys():
    df_iv = pd.read_csv(output_data_path + "A1_" + stock_ident + "_" + "Daily_vol_smile_" + prod_type + "_" + each_month + ".csv")
    df_futures = df_futures_mkt[get_mthly_df_name_from_expiry(datetime.strptime(each_month, "%d-%b-%Y").date() , "FUT", holidays_list)]

    week_id = 0
    weekly_lvols = []
    for each_week in dict_wkly_expiries_each_month[each_month]:
        
              

        
        if (each_week != \
            prev_workday_if_holiday(date_of_prev_thurs(datetime.strptime(\
            each_month, "%d-%b-%Y").date(), 1),holidays_list)):
            
            dt_mnth = float((datetime.strptime(each_month, "%d-%b-%Y").date() - each_week).days) / 365
            wk_end = prev_workday_if_holiday(each_week + timedelta(days=7), holidays_list)
            dt_wk = float((wk_end - each_week).days) / 365
            T_array = [dt_wk, dt_mnth]
            df = df_iv[df_iv["Date"] == each_week.strftime("%d-%b-%Y")]
            df = df.sort_values(by=['T'])
            df_lvol_wk = df
            
            strikes_list = [ np.array(df[round(df['T'],5) == round(dt_mnth,5)]['Strike']),  np.array(df[round(df['T'],5) == round(dt_wk,5)]['Strike'])]
            list_of_params = params_dict[each_month][week_id]
            week_id = week_id + 1
            s = df['Spot'].tolist()[0]
            
            greeks_list = generate_greeks_for_local_vol(s, strikes_list, T_array, list_of_params, prod_type ="CE")
            df_lvol_wk['dvol_by_dk'] = list(greeks_list[0]['dvol_by_dk']) + list(greeks_list[1]['dvol_by_dk'])       
            df_lvol_wk['dvol_by_dk'] = list(greeks_list[0]['dvol_by_dk']) + list(greeks_list[1]['dvol_by_dk'])
            df_lvol_wk['dvol_by_dksq'] = list(greeks_list[0]['dvol_by_dksq']) + list(greeks_list[1]['dvol_by_dksq'])
            df_lvol_wk['dvol_by_dT'] = list(greeks_list[0]['dvol_by_dT']) + list(greeks_list[1]['dvol_by_dT'])
             
        else:
            dt_mnth = float((datetime.strptime(each_month, "%d-%b-%Y").date() - each_week).days) / 365
            T_array = [dt_mnth]
            df = df_iv[df_iv["Date"] == each_week.strftime("%d-%b-%Y")]
            df_lvol_wk = df
            strikes_list = [ np.array(df[round(df['T'],5) == round(dt_mnth,5)]['Strike'])]
            list_of_params = params_dict[each_month][week_id]
            week_id = week_id + 1
            s = df['Spot'].tolist()[0]
            greeks_list = generate_greeks_for_local_vol(s, strikes_list, T_array, list_of_params, prod_type ="CE")
            df_lvol_wk = df
            df_lvol_wk['dvol_by_dk'] = list(greeks_list[0]['dvol_by_dk'])        
            df_lvol_wk['dvol_by_dk'] = list(greeks_list[0]['dvol_by_dk'])
            df_lvol_wk['dvol_by_dksq'] = list(greeks_list[0]['dvol_by_dksq'])
            df_lvol_wk['dvol_by_dT'] = list(greeks_list[0]['dvol_by_dT'])

        df_curr_futures = df_futures[df_futures['Date'] == each_week.strftime("%d-%b-%Y")]
        F_t = float(df_curr_futures['Settle Price'].tolist()[0])
        r = float(1 / dt_mnth) * np.log(F_t / s)
    
        deta_0, deta_1, deta_2, lambda_0, lambda_1, lambda_2 = list_of_params
        x1 = np.array(df_lvol_wk['Moneyness']).reshape(-1)
        x2 = np.array(df_lvol_wk['T']).reshape(-1)
        df_lvol_wk['pred_iv']= fitting_func((x1, x2), deta_0, deta_1, deta_2, lambda_0, lambda_1, lambda_2)

        df_lvol_wk['lvol'] = df_lvol_wk.apply(lambda x: \
                                              (generate_local_vol_from_impl_vol(x['Impl_Vol'], \
                                                                               x['dvol_by_dk'], \
                                                                               x['dvol_by_dksq'], \
                                                                               x['dvol_by_dT'], \
                                                                               x['Strike'], x['Spot'], x['T'], r)),\
                                                                               axis=1)
        weekly_lvols.append(df_lvol_wk)
    
    df_lvol_month = pd.concat(weekly_lvols, axis=0) 
    df_lvol_month.to_csv(output_data_path + "A2_LVOL_" + stock_ident + "_" + prod_type + "_" + each_month + ".csv", index = False)
        


/opt/tljh/user/lib/python3.7/site-packages/scipy-1.6.1-py3.7-linux-x86_64.egg/scipy/optimize/minpack.py:829: OptimizeWarning: Covariance of the parameters could not be estimated
  category=OptimizeWarning)
/opt/tljh/user/lib/python3.7/site-packages/ipykernel_launcher.py:59: RuntimeWarning: overflow encountered in square
/opt/tljh/user/lib/python3.7/site-packages/ipykernel_launcher.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/opt/tljh/user/lib/python3.7/site-packages/ipykernel_launcher.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.